# Colab Setup

In [17]:
import os
import json
import warnings
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import shutil
import subprocess

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
from torchvision import transforms

from sklearn.metrics import roc_auc_score, silhouette_score
from statsmodels.stats.contingency_tables import mcnemar
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# PATHS & CONFIGURATION

In [18]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

ARCHIVE_PATH = "/content/drive/MyDrive/COS30082/DPD_Small.7z"
EXTRACT_ROOT = "/content/data/DPD_Small_extracted"

if not os.path.exists(ARCHIVE_PATH):
    raise FileNotFoundError(f"Archive not found at {ARCHIVE_PATH}. Verify Google Drive is mounted.")

os.makedirs(EXTRACT_ROOT, exist_ok=True)
if not any(os.scandir(EXTRACT_ROOT)):
    if shutil.which("7z") is None:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", "p7zip-full"], check=True)
    print(f"Extracting {ARCHIVE_PATH} -> {EXTRACT_ROOT} ...")
    subprocess.run(["7z", "x", ARCHIVE_PATH, f"-o{EXTRACT_ROOT}", "-y"], check=True)
    print("Extraction complete.")

# Locate the 3 CSV splits and the DPD_Small_dataset image directory
TRAIN_CSV, TEST_SEEN_CSV, TEST_UNSEEN_CSV, IMG_DIR = None, None, None, None
for root, dirs, files in os.walk(EXTRACT_ROOT):
    if "DPD_Small_train.csv" in files:
        TRAIN_CSV = os.path.join(root, "DPD_Small_train.csv")
    if "DPD_Small_test_seen.csv" in files:
        TEST_SEEN_CSV = os.path.join(root, "DPD_Small_test_seen.csv")
    if "DPD_Small_test_unseen.csv" in files:
        TEST_UNSEEN_CSV = os.path.join(root, "DPD_Small_test_unseen.csv")
    if "DPD_Small_dataset" in dirs:
        IMG_DIR = os.path.join(root, "DPD_Small_dataset")

if not all([TRAIN_CSV, TEST_SEEN_CSV, TEST_UNSEEN_CSV]):
    raise FileNotFoundError(f"Missing one or more DPD_Small_*.csv files inside {EXTRACT_ROOT}")

CSV_DIR = os.path.dirname(TRAIN_CSV)
if IMG_DIR is None:
    IMG_DIR = CSV_DIR

print(f"CSVs resolved in: {CSV_DIR}")
print(f"Images resolved in: {IMG_DIR}")

BASE_OUT = "/content/drive/MyDrive/COS30082/outputs"
REPORT_DIR = os.path.join(BASE_OUT, "final_fair_evaluation")
os.makedirs(REPORT_DIR, exist_ok=True)

CHECKPOINTS = {
    "Baseline 2 (ViT-B/16)": {
        "path": os.path.join(BASE_OUT, "baseline2_multitask_vit/final/best.pt"),
        "default_bias": 4.995,
    },
    "Model 1 (Composite SupCon)": {
        "path": os.path.join(BASE_OUT, "model1_factorized_vit/model1_best.pt"),
        "default_bias": 13.750,
    },
    "Model 2 (Factorized Frozen)": {
        "path": os.path.join(BASE_OUT, "model2_factorized_vit/model2_best.pt"),
        "default_bias": 5.000,
    },
    "Model 3 (Factorized Partial-Unfreeze)": {
        "path": os.path.join(BASE_OUT, "model3_factorized_vit/model3_best.pt"),
        "default_bias": 5.324,
    },
    "Model 4 (Single-Stage Joint SupCon)": {
        "path": os.path.join(BASE_OUT, "model4_factorized_vit/model4_best.pt"),
        "default_bias": 5.500,
    },
    "Model 5 (Orthogonal Factorized SupCon)": {
        "path": os.path.join(BASE_OUT, "model5_factorized_vit/model5_best.pt"),
        "default_bias": 8.950,
    },
}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CSVs resolved in: /content/data/DPD_Small_extracted
Images resolved in: /content/data/DPD_Small_extracted/DPD_Small_dataset


# DATASET & IDENTICAL PREPROCESSING

In [19]:
CSV_COLS = ["image_path", "crop_id", "disease_id", "comp_id", "crop_name", "disease_name", "comp_name"]

train_df = pd.read_csv(TRAIN_CSV, header=None, names=CSV_COLS)
test_seen_df = pd.read_csv(TEST_SEEN_CSV, header=None, names=CSV_COLS)
test_unseen_df = pd.read_csv(TEST_UNSEEN_CSV, header=None, names=CSV_COLS)

test_seen_df["is_seen"] = True
test_unseen_df["is_seen"] = False
test_df = pd.concat([test_seen_df, test_unseen_df], ignore_index=True)

for df in (train_df, test_df):
    df["c_id"] = df["crop_id"].astype(int)
    df["d_id"] = df["disease_id"].astype(int)
    df["comp_id"] = df["comp_id"].astype(int)

NUM_CROPS = max(train_df["c_id"].max(), test_df["c_id"].max()) + 1        # 55
NUM_DISEASES = max(train_df["d_id"].max(), test_df["d_id"].max()) + 1  # 175

# Use canonical comp_id (0..332) directly from the CSVs
all_pairs_df = (
    pd.concat([train_df[["comp_id", "c_id", "d_id"]], test_df[["comp_id", "c_id", "d_id"]]])
    .drop_duplicates()
    .sort_values("comp_id")
    .reset_index(drop=True)
)

crop_of = torch.tensor(all_pairs_df["c_id"].values, dtype=torch.long)
disease_of = torch.tensor(all_pairs_df["d_id"].values, dtype=torch.long)
NUM_COMPS = len(all_pairs_df)  # 333

seen_comp_set = set(train_df["comp_id"].unique())
seen_mask = torch.tensor([cid in seen_comp_set for cid in all_pairs_df["comp_id"]], dtype=torch.bool)
unseen_mask = ~seen_mask

# Marginal log-priors from training set for Model 5 Compositional Prior Gate
crop_counts = np.bincount(train_df["c_id"].values, minlength=NUM_CROPS) + 1e-6
dis_counts = np.bincount(train_df["d_id"].values, minlength=NUM_DISEASES) + 1e-6
log_p_crop = np.log(crop_counts / crop_counts.sum())
log_p_dis = np.log(dis_counts / dis_counts.sum())
pair_prior_c = (log_p_crop[crop_of.numpy()] - log_p_crop.mean()) / (log_p_crop.std() + 1e-8)
pair_prior_d = (log_p_dis[disease_of.numpy()] - log_p_dis.mean()) / (log_p_dis.std() + 1e-8)

# Exact ViT-B/16 evaluation transform from blueprint
eval_transform = transforms.Compose([
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

class DPDTestDataset(Dataset):
    def __init__(self, df, img_dir, csv_dir, extract_root, transform):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.csv_dir = csv_dir
        self.extract_root = extract_root
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        p = str(row["image_path"])
        candidates = [
            p,
            os.path.join(self.img_dir, p),
            os.path.join(self.csv_dir, p),
            os.path.join(self.extract_root, p),
        ]
        full_path = next((c for c in candidates if os.path.exists(c)), None)
        if full_path is None:
            raise FileNotFoundError(f"Image not found for '{p}'. Checked: {candidates}")
        img = Image.open(full_path).convert("RGB")
        return self.transform(img), int(row["c_id"]), int(row["d_id"]), int(row["comp_id"]), bool(row["is_seen"])

test_loader = DataLoader(
    DPDTestDataset(test_df, IMG_DIR, CSV_DIR, EXTRACT_ROOT, eval_transform),
    batch_size=64, shuffle=False, num_workers=2, pin_memory=True
)

# ADAPTIVE MULTI-TASK VIT LOADER

In [20]:
class EvalMultiTaskViT(nn.Module):
    def __init__(self, num_crops, num_diseases, split_subspace=False, branch_dim=0):
        super().__init__()
        self.backbone = timm.create_model("vit_base_patch16_224", pretrained=False, num_classes=0)
        self.split_subspace = split_subspace
        self.branch_dim = branch_dim

        if split_subspace:
            # Model 5: first 384 dims -> crop_head, last 384 dims -> disease_head
            self.sub_dim = 384
            self.crop_head = nn.Linear(384, num_crops)
            self.disease_head = nn.Linear(384, num_diseases)
        elif branch_dim > 0:
            # Model 4 with optional branches
            self.crop_branch = nn.Sequential(nn.Linear(768, branch_dim), nn.LayerNorm(branch_dim), nn.GELU())
            self.disease_branch = nn.Sequential(nn.Linear(768, branch_dim), nn.LayerNorm(branch_dim), nn.GELU())
            self.crop_head = nn.Linear(branch_dim, num_crops)
            self.disease_head = nn.Linear(branch_dim, num_diseases)
        else:
            # Baseline 2 & Models 1-4
            self.crop_head = nn.Linear(768, num_crops)
            self.disease_head = nn.Linear(768, num_diseases)

    def forward(self, x):
        feat = self.backbone(x)  # [B, 768] [CLS] embedding
        if self.split_subspace:
            return self.crop_head(feat[:, :self.sub_dim]), self.disease_head(feat[:, self.sub_dim:]), feat
        if self.branch_dim > 0:
            return self.crop_head(self.crop_branch(feat)), self.disease_head(self.disease_branch(feat)), feat
        return self.crop_head(feat), self.disease_head(feat), feat

def load_checkpoint_clean(ckpt_path, default_bias):
    raw = torch.load(ckpt_path, map_location="cpu")
    bias = default_bias
    if isinstance(raw, dict):
        for b_key in ["dev_bias", "best_bias", "bias", "optimal_bias"]:
            if b_key in raw and raw[b_key] is not None:
                bias = float(raw[b_key])
                break
        state = raw.get("model_state_dict", raw.get("state_dict", raw.get("model", raw)))
    else:
        state = raw

    cleaned = {}
    for k, v in state.items():
        new_k = k.replace("module.", "")
        if new_k.startswith("encoder."):
            new_k = "backbone." + new_k[len("encoder."):]
        elif new_k.startswith("vit."):
            new_k = "backbone." + new_k[len("vit."):]
        elif new_k.startswith("backbone.model."):
            new_k = "backbone." + new_k[len("backbone.model."):]
        # Normalize Model 2 & Model 3 head names
        if new_k.startswith("head_crop."):
            new_k = "crop_head." + new_k[len("head_crop."):]
        elif new_k.startswith("head_disease."):
            new_k = "disease_head." + new_k[len("head_disease."):]
        # Skip Stage-1 contrastive projection heads
        if any(skip in new_k for skip in ["projector", "proj_head", "crop_proj", "disease_proj", "dis_proj", "comp_head"]):
            continue
        cleaned[new_k] = v

    in_dim = cleaned["crop_head.weight"].shape[1]
    has_branches = "crop_branch.0.weight" in cleaned
    split_subspace = (in_dim == 384) and not has_branches
    branch_dim = in_dim if has_branches else 0

    model = EvalMultiTaskViT(NUM_CROPS, NUM_DISEASES, split_subspace=split_subspace, branch_dim=branch_dim)
    missing, unexpected = model.load_state_dict(cleaned, strict=False)
    print(f"Loaded {os.path.basename(ckpt_path)} | dev_bias={bias:.3f} | missing={len(missing)} | unexpected={len(unexpected)}")
    return model.to(device).eval(), bias

# INFERENCE & GZSL METRIC ENGINE

In [21]:
@torch.no_grad()
def extract_logits_and_features(model):
    all_c_logits, all_d_logits, all_feats = [], [], []
    all_c_true, all_d_true, all_comp_true, all_is_seen = [], [], [], []

    for imgs, c_id, d_id, comp_id, is_seen in test_loader:
        imgs = imgs.to(device, non_blocking=True)
        with torch.cuda.amp.autocast():
            c_log, d_log, feat = model(imgs)
        all_c_logits.append(c_log.float().cpu())
        all_d_logits.append(d_log.float().cpu())
        all_feats.append(F.normalize(feat.float(), dim=-1).cpu())
        all_c_true.append(c_id)
        all_d_true.append(d_id)
        all_comp_true.append(comp_id)
        all_is_seen.append(is_seen)

    return {
        "c_logits": torch.cat(all_c_logits, dim=0),
        "d_logits": torch.cat(all_d_logits, dim=0),
        "feats": torch.cat(all_feats, dim=0).numpy(),
        "c_true": torch.cat(all_c_true, dim=0),
        "d_true": torch.cat(all_d_true, dim=0),
        "comp_true": torch.cat(all_comp_true, dim=0),
        "is_seen": torch.cat(all_is_seen, dim=0).numpy(),
    }

def evaluate_predictions(c_logits, d_logits, c_true, d_true, comp_true, is_seen, bias_vector):
    """Evaluates composition & head metrics given a per-composition bias vector [NUM_COMPS]."""
    base_scores = c_logits[:, crop_of] + d_logits[:, disease_of]  # [N, 333]
    scores = base_scores - bias_vector.unsqueeze(0)

    seen_idx = np.where(is_seen)[0]
    unseen_idx = np.where(~is_seen)[0]

    # Top-1 & Top-5 predictions
    top5_preds = torch.topk(scores, k=5, dim=1).indices
    top1_preds = top5_preds[:, 0]

    correct_top1 = (top1_preds == comp_true).numpy()
    correct_top5 = (top5_preds == comp_true.unsqueeze(1)).any(dim=1).numpy()

    seen_top1 = correct_top1[seen_idx].mean() * 100
    unseen_top1 = correct_top1[unseen_idx].mean() * 100
    seen_top5 = correct_top5[seen_idx].mean() * 100
    unseen_top5 = correct_top5[unseen_idx].mean() * 100
    hm = (2 * seen_top1 * unseen_top1) / (seen_top1 + unseen_top1 + 1e-8)

    # Shortcut rate: % of unseen test samples predicted as a SEEN composition
    pred_is_seen = seen_mask[top1_preds].numpy()
    shortcut_rate = pred_is_seen[unseen_idx].mean() * 100

    # Closed-world unseen ceiling (b = infinity -> mask out seen pairs)
    cw_scores = base_scores.clone()
    cw_scores[:, seen_mask] = -1e9
    cw_top1 = (cw_scores.argmax(dim=1) == comp_true).numpy()[unseen_idx].mean() * 100

    # Marginal head accuracies
    c_pred = c_logits.argmax(dim=1)
    d_pred = d_logits.argmax(dim=1)
    seen_crop_acc = (c_pred == c_true).numpy()[seen_idx].mean() * 100
    seen_dis_acc = (d_pred == d_true).numpy()[seen_idx].mean() * 100
    unseen_crop_acc = (c_pred == c_true).numpy()[unseen_idx].mean() * 100
    unseen_dis_acc = (d_pred == d_true).numpy()[unseen_idx].mean() * 100

    # OOD AUROC (max unseen score - max seen score as novelty confidence)
    ood_score = (base_scores[:, unseen_mask].max(dim=1).values - base_scores[:, seen_mask].max(dim=1).values).numpy()
    ood_auroc = roc_auc_score(~is_seen, ood_score)

    return {
        "Seen@1 (%)": seen_top1,
        "Unseen@1 (%)": unseen_top1,
        "Calibrated HM (%)": hm,
        "Seen@5 (%)": seen_top5,
        "Unseen@5 (%)": unseen_top5,
        "Shortcut Rate (%)": shortcut_rate,
        "Unseen Ceiling (%)": cw_top1,
        "Seen Crop (%)": seen_crop_acc,
        "Seen Disease (%)": seen_dis_acc,
        "Unseen Crop (%)": unseen_crop_acc,
        "Unseen Disease (%)": unseen_dis_acc,
        "OOD AUROC": ood_auroc,
        "correct_top1": correct_top1,
        "top1_preds": top1_preds.numpy(),
    }

def sweep_bias_curve(c_logits, d_logits, comp_true, is_seen):
    base_scores = c_logits[:, crop_of] + d_logits[:, disease_of]
    seen_idx = np.where(is_seen)[0]
    unseen_idx = np.where(~is_seen)[0]

    biases = np.linspace(-15.0, 25.0, 201)
    seen_accs, unseen_accs, hms = [], [], []
    for b in biases:
        s = base_scores.clone()
        s[:, seen_mask] -= b
        preds = s.argmax(dim=1)
        corr = (preds == comp_true).numpy()
        sa = corr[seen_idx].mean() * 100
        ua = corr[unseen_idx].mean() * 100
        seen_accs.append(sa)
        unseen_accs.append(ua)
        hms.append((2 * sa * ua) / (sa + ua + 1e-8))

    # Sort by Seen accuracy ascending to integrate Seen-Unseen AUC cleanly
    order = np.argsort(seen_accs)
    trapz_fn = getattr(np, "trapezoid", None) or np.trapz
    auc = trapz_fn(np.array(unseen_accs)[order] / 100.0, np.array(seen_accs)[order] / 100.0)
    best_idx = int(np.argmax(hms))
    return {
        "biases": biases,
        "seen": np.array(seen_accs),
        "unseen": np.array(unseen_accs),
        "hm": np.array(hms),
        "auc": auc,
        "oracle_hm": hms[best_idx],
        "oracle_bias": biases[best_idx],
    }

# RUN UNIFIED EVALUATION ACROSS ALL MODELS

In [23]:
results_rows = []
sweep_data = {}
raw_outputs = {}

for name, cfg in CHECKPOINTS.items():
    model, dev_b = load_checkpoint_clean(cfg["path"], cfg["default_bias"])
    out = extract_logits_and_features(model)
    raw_outputs[name] = out

    # 1. Calibrated evaluation (using dev_bias)
    b_vec = torch.zeros(NUM_COMPS)
    b_vec[seen_mask] = dev_b
    cal = evaluate_predictions(out["c_logits"], out["d_logits"], out["c_true"], out["d_true"], out["comp_true"], out["is_seen"], b_vec)

    # 2. Uncalibrated evaluation (b = 0)
    uncal = evaluate_predictions(out["c_logits"], out["d_logits"], out["c_true"], out["d_true"], out["comp_true"], out["is_seen"], torch.zeros(NUM_COMPS))

    # 3. Bias sweep (AUC & Oracle HM)
    sw = sweep_bias_curve(out["c_logits"], out["d_logits"], out["comp_true"], out["is_seen"])
    sweep_data[name] = sw

    # 4. Latent space Silhouette Scores (subset 2,500 samples for speed)
    sub = np.random.RandomState(42).choice(len(out["feats"]), size=min(2500, len(out["feats"])), replace=False)
    sil_crop = silhouette_score(out["feats"][sub], out["c_true"].numpy()[sub])
    sil_dis = silhouette_score(out["feats"][sub], out["d_true"].numpy()[sub])

    results_rows.append({
        "Model": name,
        "Dev Bias (b)": round(dev_b, 3),
        "Seen@1 (%)": round(cal["Seen@1 (%)"], 2),
        "Unseen@1 (%)": round(cal["Unseen@1 (%)"], 2),
        "Calibrated HM (%)": round(cal["Calibrated HM (%)"], 2),
        "Oracle HM (%)": round(sw["oracle_hm"], 2),
        "Bias Gap (Oracle-Cal)": round(sw["oracle_hm"] - cal["Calibrated HM (%)"], 2),
        "Seen-Unseen AUC": round(sw["auc"], 4),
        "Uncalibrated HM (b=0)": round(uncal["Calibrated HM (%)"], 2),
        "Seen@5 (%)": round(cal["Seen@5 (%)"], 2),
        "Unseen@5 (%)": round(cal["Unseen@5 (%)"], 2),
        "Shortcut Rate (%)": round(cal["Shortcut Rate (%)"], 2),
        "Unseen Ceiling (%)": round(cal["Unseen Ceiling (%)"], 2),
        "Seen Crop (%)": round(cal["Seen Crop (%)"], 2),
        "Seen Disease (%)": round(cal["Seen Disease (%)"], 2),
        "Unseen Crop (%)": round(cal["Unseen Crop (%)"], 2),
        "Unseen Disease (%)": round(cal["Unseen Disease (%)"], 2),
        "OOD AUROC": round(cal["OOD AUROC"], 4),
        "Crop Silhouette": round(sil_crop, 4),
        "Disease Silhouette": round(sil_dis, 4),
        "_correct_top1": cal["correct_top1"],
    })

# 6. Add Model 5 Fine-Tuned (Compositional Prior Gate: b0=8.95, w=[-0.5, +1.0])
m5_out = raw_outputs["Model 5 (Orthogonal Factorized SupCon)"]
gate_bias = torch.zeros(NUM_COMPS)
gate_bias[seen_mask] = torch.tensor(
    8.950 + (-0.5 * pair_prior_c[seen_mask.numpy()]) + (1.0 * pair_prior_d[seen_mask.numpy()]),
    dtype=torch.float32
)
m5_tuned = evaluate_predictions(
    m5_out["c_logits"], m5_out["d_logits"], m5_out["c_true"], m5_out["d_true"], m5_out["comp_true"], m5_out["is_seen"], gate_bias
)
m5_sw = sweep_data["Model 5 (Orthogonal Factorized SupCon)"]
m5_base_row = results_rows[-1]

results_rows.append({
    "Model": "Model 5 + Compositional Prior Gate (Tuned)",
    "Dev Bias (b)": "8.95 + PriorGate",
    "Seen@1 (%)": round(m5_tuned["Seen@1 (%)"], 2),
    "Unseen@1 (%)": round(m5_tuned["Unseen@1 (%)"], 2),
    "Calibrated HM (%)": round(m5_tuned["Calibrated HM (%)"], 2),
    "Oracle HM (%)": m5_base_row["Oracle HM (%)"],
    "Bias Gap (Oracle-Cal)": round(m5_base_row["Oracle HM (%)"] - m5_tuned["Calibrated HM (%)"], 2),
    "Seen-Unseen AUC": m5_base_row["Seen-Unseen AUC"],
    "Uncalibrated HM (b=0)": m5_base_row["Uncalibrated HM (b=0)"],
    "Seen@5 (%)": round(m5_tuned["Seen@5 (%)"], 2),
    "Unseen@5 (%)": round(m5_tuned["Unseen@5 (%)"], 2),
    "Shortcut Rate (%)": round(m5_tuned["Shortcut Rate (%)"], 2),
    "Unseen Ceiling (%)": round(m5_tuned["Unseen Ceiling (%)"], 2),
    "Seen Crop (%)": round(m5_tuned["Seen Crop (%)"], 2),
    "Seen Disease (%)": round(m5_tuned["Seen Disease (%)"], 2),
    "Unseen Crop (%)": round(m5_tuned["Unseen Crop (%)"], 2),
    "Unseen Disease (%)": round(m5_tuned["Unseen Disease (%)"], 2),
    "OOD AUROC": round(m5_tuned["OOD AUROC"], 4),
    "Crop Silhouette": m5_base_row["Crop Silhouette"],
    "Disease Silhouette": m5_base_row["Disease Silhouette"],
    "_correct_top1": m5_tuned["correct_top1"],
})

Loaded best.pt | dev_bias=4.995 | missing=0 | unexpected=0
Loaded model1_best.pt | dev_bias=13.750 | missing=0 | unexpected=0
Loaded model2_best.pt | dev_bias=5.421 | missing=0 | unexpected=0
Loaded model3_best.pt | dev_bias=5.324 | missing=0 | unexpected=0
Loaded model4_best.pt | dev_bias=5.314 | missing=0 | unexpected=0
Loaded model5_best.pt | dev_bias=8.950 | missing=0 | unexpected=0


# STATISTICAL SIGNIFICANCE (MCNEMAR TEST VS BASELINE 2)

In [24]:
base_correct = results_rows[0]["_correct_top1"]
is_seen_mask = raw_outputs["Baseline 2 (ViT-B/16)"]["is_seen"]

for row in results_rows:
    cand_correct = row["_correct_top1"]
    # McNemar on Unseen Test Split
    u_base = base_correct[~is_seen_mask]
    u_cand = cand_correct[~is_seen_mask]
    b01 = np.sum((u_base == 0) & (u_cand == 1))  # Baseline wrong, Model right
    b10 = np.sum((u_base == 1) & (u_cand == 0))  # Baseline right, Model wrong
    if (b01 + b10) > 0:
        res = mcnemar([[np.sum((u_base == 1) & (u_cand == 1)), b10],
                       [b01, np.sum((u_base == 0) & (u_cand == 0))]], exact=False)
        row["Unseen McNemar p-val"] = f"{res.pvalue:.2e}"
    else:
        row["Unseen McNemar p-val"] = "1.00"
    del row["_correct_top1"]

summary_df = pd.DataFrame(results_rows)
csv_path = os.path.join(REPORT_DIR, "master_fair_comparison.csv")
summary_df.to_csv(csv_path, index=False)
print("\n=== MASTER FAIR COMPARISON TABLE ===")
print(summary_df.to_string(index=False))



=== MASTER FAIR COMPARISON TABLE ===
                                     Model     Dev Bias (b)  Seen@1 (%)  Unseen@1 (%)  Calibrated HM (%)  Oracle HM (%)  Bias Gap (Oracle-Cal)  Seen-Unseen AUC  Uncalibrated HM (b=0)  Seen@5 (%)  Unseen@5 (%)  Shortcut Rate (%)  Unseen Ceiling (%)  Seen Crop (%)  Seen Disease (%)  Unseen Crop (%)  Unseen Disease (%)  OOD AUROC  Crop Silhouette  Disease Silhouette Unseen McNemar p-val
                     Baseline 2 (ViT-B/16)            4.995       65.60         54.21              59.37          64.24                   4.87           0.5382                   9.65       89.80         94.13              32.87               82.39           93.0             72.10            77.42                9.32     0.8866           0.0801             -0.0044                 1.00
                Model 1 (Composite SupCon)            13.75       32.90         79.14              46.48          59.81                  13.33           0.4530                   6.44      

# GENERATE REPORT FIGURES

In [25]:
plt.style.use("seaborn-v0_8-whitegrid")

# Figure 1: Seen-Unseen Pareto Calibration Curves
fig, ax = plt.subplots(figsize=(9, 6), dpi=180)
for name, sw in sweep_data.items():
    lw = 2.8 if "Model 5" in name or "Baseline 2" in name else 1.6
    ls = "--" if "Baseline 2" in name else "-"
    ax.plot(sw["seen"], sw["unseen"], label=f"{name} (AUC={sw['auc']:.3f})", linewidth=lw, linestyle=ls)

# Highlight calibrated operating points
for _, r in summary_df.iterrows():
    marker = "*" if "Tuned" in r["Model"] else "o"
    size = 160 if "Tuned" in r["Model"] else 65
    ax.scatter(r["Seen@1 (%)"], r["Unseen@1 (%)"], s=size, marker=marker, zorder=5)

ax.set_xlabel("Seen Top-1 Accuracy (%)", fontsize=11, fontweight="bold")
ax.set_ylabel("Unseen Top-1 Accuracy (%)", fontsize=11, fontweight="bold")
ax.set_title("Generalized Zero-Shot Trade-Off Curves (Seen vs. Unseen Accuracy)", fontsize=13, fontweight="bold")
ax.legend(loc="lower left", fontsize=8.5, frameon=True)
plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, "fig1_seen_unseen_pareto.png"))
plt.close()

# Figure 2: Diagnostic Decomposition (Calibrated HM vs Oracle HM & Unseen Disease)
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), dpi=180)
short_names = ["Baseline 2\n(ViT)", "Model 1\n(Comp)", "Model 2\n(Fact-Frz)", "Model 3\n(Unfreeze)", "Model 4\n(1-Stage)", "Model 5\n(Orthog)", "Model 5\n+PriorGate"]
x = np.arange(len(short_names))

axes[0].bar(x - 0.18, summary_df["Calibrated HM (%)"], width=0.35, label="Deployed / Calibrated HM", color="#2b5c8f")
axes[0].bar(x + 0.18, summary_df["Oracle HM (%)"], width=0.35, label="Oracle HM (Upper Bound)", color="#8fb8de")
axes[0].set_xticks(x)
axes[0].set_xticklabels(short_names, fontsize=8.5)
axes[0].set_ylabel("Harmonic Mean (%)")
axes[0].set_title("Calibrated vs. Oracle Harmonic Mean (Bias Sensitivity)", fontweight="bold")
axes[0].legend()

axes[1].bar(x - 0.18, summary_df["Unseen Disease (%)"], width=0.35, label="Unseen Disease Top-1 (%)", color="#d95f02")
axes[1].bar(x + 0.18, summary_df["Unseen Crop (%)"] / 5.0, width=0.35, label="Unseen Crop Top-1 (÷5 scale)", color="#7570b3")
axes[1].set_xticks(x)
axes[1].set_xticklabels(short_names, fontsize=8.5)
axes[1].set_ylabel("Accuracy (%)")
axes[1].set_title("Compositional Bottleneck: Unseen Disease Disentanglement", fontweight="bold")
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, "fig2_diagnostic_decomposition.png"))
plt.close()

print(f"\nAll tables and figures saved to: {REPORT_DIR}")


All tables and figures saved to: /content/drive/MyDrive/COS30082/outputs/final_fair_evaluation
